# 12장 보안 실습 — 오프라인 DFIR Capstone


## Goal

고정 자료의 사본 수집·상태·해시와 사실/가설 인계를 통합합니다.

[교안과 분석 질문](../../12-capstone/12-3-dfir-capstone.md)을 먼저 읽습니다.


## Setup

Python 커널의 %%bash를 사용합니다. 새 임시 폴더에 합성 자료와 결과 경로를 준비합니다. 외부 접속·서비스 등록·원본 서버 조사는 하지 않습니다. 코드를 검토하고 Setup부터 순서대로 실행합니다. Bash 셀 사이의 상태는 환경 변수와 파일로 전달합니다.


In [ ]:
from pathlib import Path
import hashlib
import os
import tempfile

lab = Path(tempfile.mkdtemp(prefix='bash-security-12-'))
data = lab / 'data'
output = lab / 'output'
data.mkdir()
output.mkdir()
fixtures = {'access.log': '192.0.2.10 - - [10/Sep/2026:09:01:00 +0900] "POST /login HTTP/1.1" 401 120 "-" "CourseBrowser/1.0"\n192.0.2.10 - - [10/Sep/2026:09:01:10 +0900] "POST /login HTTP/1.1" 401 120 "-" "CourseBrowser/1.0"\n198.51.100.8 - - [10/Sep/2026:09:01:20 +0900] "POST /login HTTP/1.1" 401 120 "-" "CourseBrowser/1.0"\n203.0.113.7 - - [10/Sep/2026:09:04:00 +0900] "GET /health HTTP/1.1" 200 12 "-" "CourseMonitor/1.0"\n203.0.113.7 - - [10/Sep/2026:09:04:10 +0900] "GET /uploads/report.txt HTTP/1.1" 200 45 "-" "CourseBrowser/1.0"\n192.0.2.10 - - [10/Sep/2026:09:04:20 +0900] "GET /admin HTTP/1.1" 404 80 "-" "CourseBrowser/1.0"\n', 'audit.log': 'type=SYSCALL msg=audit(1788998580.000:900): arch=c000003e syscall=59 success=yes exit=0 a0=0 a1=0 a2=0 a3=0 items=1 ppid=410 pid=450 auid=1000 uid=0 gid=0 euid=0 suid=0 fsuid=0 egid=0 sgid=0 fsgid=0 tty=pts0 ses=4 comm="id" exe="/usr/bin/id" key="course_exec"\ntype=EXECVE msg=audit(1788998580.000:900): argc=1 a0="id"\ntype=CWD msg=audit(1788998580.000:900): cwd="/home/analyst"\ntype=PATH msg=audit(1788998580.000:900): item=0 name="/usr/bin/id" inode=100 dev=08:01 mode=0100755 ouid=0 ogid=0 rdev=00:00 nametype=NORMAL\ntype=PROCTITLE msg=audit(1788998580.000:900): proctitle=6964\ntype=EOE msg=audit(1788998580.000:900):\n', 'auth.log': '2026-09-10T09:01:00+09:00 lab-web-01 sshd[101]: Failed password for invalid user guest from 192.0.2.10 port 50100 ssh2\n2026-09-10T09:01:10+09:00 lab-web-01 sshd[102]: Failed password for analyst from 192.0.2.10 port 50101 ssh2\n2026-09-10T09:01:20+09:00 lab-web-01 sshd[103]: Failed password for analyst from 198.51.100.8 port 50102 ssh2\n2026-09-10T09:02:00+09:00 lab-web-01 sshd[104]: Accepted publickey for analyst from 192.0.2.10 port 50103 ssh2\n2026-09-10T09:03:00+09:00 lab-web-01 sudo: analyst : TTY=pts/0 ; PWD=/home/analyst ; USER=root ; COMMAND=/usr/bin/id\n', 'ioc.log': '2026-09-10T09:01:00+09:00 src=192.0.2.10 action=failed\n2026-09-10T09:02:00+09:00 src=192.0.2.100 action=accepted\n2026-09-10T09:03:00+09:00 note=indicator.example\n2026-09-10T09:04:00+09:00 note=indicatorXexample\n', 'journal-review.psv': 'time_kst|boot_id|unit|pid|message\n2026-09-10T08:00:00+09:00|BOOT-A|init.scope|1|System boot\n2026-09-10T09:02:00+09:00|BOOT-A|ssh.service|104|Accepted publickey for analyst\n2026-09-10T09:05:00+09:00|BOOT-A|report-helper.service|1|Started report helper\n2026-09-10T09:06:00+09:00|BOOT-A|report-helper.service|520|Report completed\n', 'login-review.psv': 'artifact|user|source|time_kst|meaning\nbtmp-summary|analyst|192.0.2.10|2026-09-10T09:01:10+09:00|failed\nwtmp-summary|analyst|192.0.2.10|2026-09-10T09:02:00+09:00|session-start\nlastlog-summary|analyst|192.0.2.10|2026-09-10T09:02:00+09:00|latest-login\nutmp-summary|analyst|192.0.2.10|2026-09-10T09:10:00+09:00|present-at-collection\n', 'passwd.sample': 'root:x:0:0:root:/root:/bin/bash\nanalyst:x:1000:1000:Analyst:/home/analyst:/bin/bash\ncollector:x:995:995:Collector:/var/lib/collector:/usr/sbin/nologin\nlegacy-admin:x:0:0:Legacy:/var/lib/legacy:/usr/sbin/nologin\n', 'permissions.psv': 'path|owner|group|mode|purpose|approval\n/usr/bin/passwd|root|root|4755|password-management|baseline\n/opt/collector/bin/report|root|collector|0775|service-executable|review\n/var/tmp/course-cache|root|root|1777|shared-temp|baseline\n', 'persistence.psv': 'mechanism|path|owner|approval|observed_kst\nsystemd|/etc/systemd/system/report-helper.service|root|unknown|2026-09-10T09:05:00+09:00\ncron|/etc/cron.d/backup|root|CHG-100|2026-09-09T18:00:00+09:00\nshell-startup|/home/analyst/.bashrc|analyst|baseline|2026-09-01T10:00:00+09:00\nssh-key|/home/analyst/.ssh/authorized_keys|analyst|KEY-200|2026-09-01T10:00:00+09:00\n', 'processes.psv': 'pid|ppid|user|started_kst|exe|unit\n1|0|root|2026-09-10T08:00:00+09:00|/usr/lib/systemd/systemd|init.scope\n200|1|root|2026-09-10T08:01:00+09:00|/usr/sbin/sshd|ssh.service\n410|200|analyst|2026-09-10T09:02:00+09:00|/usr/bin/bash|session-4.scope\n520|1|collector|2026-09-10T09:05:00+09:00|/opt/collector/bin/report|report-helper.service\n', 'provenance.txt': 'case_id=COURSE-IR-002\nsource_type=synthetic\nsource_host=lab-web-01\nsource_timezone=Asia/Seoul\nwindow_start=2026-09-10T09:00:00+09:00\nwindow_end=2026-09-10T10:00:00+09:00\ncollector=course-author\ncollection_scope=selected teaching records only\naudit_coverage=partial\nauthorization=offline classroom analysis\n', 'service-review.txt': '# Synthetic review excerpt only. Not an installable unit file.\nunit=report-helper.service\nUser=collector\nExecStart=/opt/collector/bin/report\nFragmentPath=/etc/systemd/system/report-helper.service\nDropInPaths=not_collected\nchange_ticket=unknown\n', 'sockets.psv': 'state|local|peer|pid\nLISTEN|0.0.0.0:22|0.0.0.0:*|200\nESTAB|192.0.2.20:22|192.0.2.10:50103|200\nESTAB|192.0.2.20:41000|203.0.113.7:443|520\n'}
for name, content in fixtures.items():
    path = data / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
before = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
tools_dir = lab / 'tools'
tools_dir.mkdir()
os.environ['COURSE_TOOLS'] = str(tools_dir)
os.environ['COURSE_DATA'] = str(data)
os.environ['COURSE_OUT'] = str(output)
print('합성 자료와 새 결과 폴더 준비 완료')


## Steps

예상 결과: complete, manifest 헤더+12항목, 기존 결과 보존, 사실2/가설1/미확인1

명령을 실행하기 전에 입력·출력·실패 조건을 표시합니다. 자료의 상세 필드 해석과 정상 행위 대안은 연결된 교안에서 확인합니다.


### 제공 도구 준비

아래 구현을 읽고 입력 검증·실패 처리·출력 경계를 표시합니다. 실행 중 다운로드하지 않으며 실습 폴더에만 저장합니다.


In [ ]:
%%bash
set -euo pipefail
cat > "$COURSE_TOOLS/triage-offline.sh" <<'COURSE_TOOL'
#!/usr/bin/env bash
# Collect a fixed allowlist of classroom evidence copies; no live host queries.
set -u
usage() { printf 'usage: bash triage-offline.sh -d DATA_DIR -o NEW_OUTPUT_DIR [-n]\n' >&2; }
data_dir='' output_dir='' dry_run=0 seen_data=0 seen_output=0
while getopts ':d:o:n' option; do
    case $option in
        d) (( seen_data == 0 )) || { usage; exit 2; }; data_dir=$OPTARG; seen_data=1 ;;
        o) (( seen_output == 0 )) || { usage; exit 2; }; output_dir=$OPTARG; seen_output=1 ;;
        n) dry_run=1 ;;
        *) usage; exit 2 ;;
    esac
done
shift "$((OPTIND - 1))"
if (( $# || !seen_data || !seen_output )) || [[ -z $data_dir || -z $output_dir ]]; then usage; exit 2; fi
if [[ ! -d $data_dir || -L $data_dir || -e $output_dir || -L $output_dir ]]; then
    printf 'invalid source directory or output already exists\n' >&2; exit 2
fi
for required in provenance.txt auth.log; do
    if [[ ! -f $data_dir/$required || ! -r $data_dir/$required || -L $data_dir/$required ]]; then
        printf 'required input unavailable: %s\n' "$required" >&2; exit 2
    fi
done
command -v python3 >/dev/null || { printf 'python3 required\n' >&2; exit 2; }
python3 - "$data_dir" "$output_dir" <<'PY' || exit 2
from pathlib import Path
import sys
source, destination = (Path(value).resolve() for value in sys.argv[1:])
if source == destination or source in destination.parents:
    print('output must be outside the source directory', file=sys.stderr)
    raise SystemExit(1)
PY
if (( dry_run )); then
    printf 'dry_run=validated required inputs; optional availability checked during collection\n'
    exit 0
fi
observed_at=$(TZ=Asia/Seoul date '+%Y-%m-%dT%H:%M:%S%z') || exit 2
case $observed_at in *+0900) ;; *) printf 'Asia/Seoul tzdata required\n' >&2; exit 2 ;; esac
umask 077
mkdir -- "$output_dir" || exit 2
printf 'category\tfile\tstatus\n' > "$output_dir/manifest.tsv" || exit 1
partial=0
collect_file() {
    local category=$1 name=$2 status
    if [[ ! -f $data_dir/$name || ! -r $data_dir/$name || -L $data_dir/$name ]]; then
        status=unavailable; partial=1
    elif cp -- "$data_dir/$name" "$output_dir/$name"; then
        status=copied
    else
        status=failed; partial=1
    fi
    printf '%s\t%s\t%s\n' "$category" "$name" "$status" >> "$output_dir/manifest.tsv" || return 1
}
collect_system() { collect_file system provenance.txt; }
collect_users() { collect_file users passwd.sample && collect_file permissions permissions.psv; }
collect_processes() { collect_file process processes.psv; }
collect_network() { collect_file network sockets.psv; }
collect_persistence() {
    collect_file persistence persistence.psv && collect_file persistence service-review.txt
}
collect_logs() {
    local name
    for name in auth.log login-review.psv journal-review.psv audit.log access.log; do
        collect_file logs "$name" || return 1
    done
}
collect_system && collect_users && collect_processes && collect_network && collect_persistence && collect_logs || exit 1
printf 'source_type=offline_classroom_copy\ncollected_at_kst=%s\n' "$observed_at" > "$output_dir/context.txt" || exit 1
if (( partial )); then completion=partial; else completion=complete; fi
printf 'collection_status=%s\n' "$completion" >> "$output_dir/context.txt" || exit 1
python3 - "$output_dir" <<'PY' || exit 1
import hashlib
from pathlib import Path
import sys
directory = Path(sys.argv[1])
lines = [f'{hashlib.sha256(p.read_bytes()).hexdigest()}  {p.name}\n'
         for p in sorted(directory.iterdir()) if p.is_file()]
with (directory / 'SHA256SUMS').open('x', encoding='utf-8') as stream:
    stream.writelines(lines)
PY
printf '%s\n' "$completion" > "$output_dir/COLLECTION_FINISHED" || exit 1
printf 'collection_status=%s\n' "$completion"
exit "$partial"
COURSE_TOOL


### 1. dry-run으로 필수 입력과 출력 경계 확인


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
bash "$COURSE_TOOLS/triage-offline.sh" -d "$COURSE_DATA" -o "$COURSE_OUT/report" -n
test ! -e "$COURSE_OUT/report"


### 2. 기능별 사본 수집과 상태 기록


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
bash "$COURSE_TOOLS/triage-offline.sh" -d "$COURSE_DATA" -o "$COURSE_OUT/report"
grep -Fx 'collection_status=complete' "$COURSE_OUT/report/context.txt"
test "$(wc -l < "$COURSE_OUT/report/manifest.tsv")" -eq 13
test -s "$COURSE_OUT/report/SHA256SUMS"
test "$(cat "$COURSE_OUT/report/COLLECTION_FINISHED")" = complete


### 3. 기존 결과 덮어쓰기 거부


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
status=0
bash "$COURSE_TOOLS/triage-offline.sh" -d "$COURSE_DATA" -o "$COURSE_OUT/report" \
 > "$COURSE_OUT/retry-out.txt" 2> "$COURSE_OUT/retry-err.txt" || status=$?
test "$status" -eq 2
test ! -s "$COURSE_OUT/retry-out.txt"
printf 'existing_report_preserved=yes\n'


### 4. 사실과 가설을 분리한 인계 기록


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
printf 'fact=failed SSH records: 3\nfact=publickey success records: 1\nhypothesis=account access needs review\nunknown=service approval and audit completeness\n' > "$COURSE_OUT/handoff.txt"
test "$(wc -l < "$COURSE_OUT/handoff.txt")" -eq 4
printf 'handoff=2 facts, 1 hypothesis, 1 unknown\n'


## Red Team ↔ Blue Team 사례 분석

### Capstone 역할 교대 — 하나의 자료 묶음, 두 개의 검토서

두 학생이 같은 합성 사본을 사용합니다. 한 명은 **Red Team 위험 검토자**로 목적·조건·영향 가설을 적고, 다른 한 명은 **Blue Team 조사자**로 실제 근거·반례·누락·개선을 적습니다. 이후 역할을 바꾸어 상대의 결론이 자료보다 앞서가는 부분을 찾습니다. 혼자 학습할 때는 두 열을 차례로 작성합니다.

| 연결 단계 | 종합 과제 |
|---|---|
| Goal / Boundary | 계정 접근·업무 도구 위임·서비스 통제라는 세 경계를 각각 설명 |
| Command / Observation | 05장 인증 집계, 06장 프로세스/소켓 조인, 07장 권한 검토, 10장 Audit 이벤트 연결 |
| System Change / Artifact | 로그인·실행·설정·파일의 현재 관찰과 과거 변경을 분리 |
| Log prerequisite | 출처·기간·KST·부분 수집과 실제 기록 기능의 공백을 명시 |
| Blue Team Investigation | 인증 성공·sudo·id·서비스·통신의 연결된 사실과 연결 미확인 부분을 구분 |
| Detection | 정상 작업·검토 필요·자료 부족 세 종류로 탐지 요구사항을 평가 |
| Mitigation | 근거가 있는 권한·구성·로깅 개선을 제안하고 보존·승인·복구 영향을 적음 |

### 제출할 사건 카드 세 개

| 카드 | Red Team 질문 | Blue Team 확인 자료 | 현재 결론의 상한 |
|---|---|---|---|
| C01 인증 | 인증·접근 정책의 경계는 적절한가? | auth.log·로그인 요약·키/장치 승인 | 실패와 공개키 성공 관찰, 비밀번호 공격 성공 미입증 |
| C02 권한 | 업무 도구 위임이 필요한 범위에 한정되는가? | 계정·권한 사본·sudo·Audit·정책/업무 문서 | id 권한 실행 관찰, 위임 정책 전체·승인 미확인 |
| C03 지속성·통신 | 서비스 구성과 목적지 통제가 적절한가? | 프로세스·소켓·서비스·Journal·변경 승인 | 서비스·연결 관찰, 악성 설치·C2 미입증 |

C02는 [GTFOBins 해설](../../07-secure-scripting/07-4-gtfobins-review.md)의 **기능 / 권한 문맥 / 업무 범위 / 실행 근거** 구분을 적용합니다. 사건의 실제 sudo 정책은 제공되지 않았으므로 임의의 취약 정책을 만들어 보고하지 않습니다. tool-review.psv는 07장 별도 사고 훈련용 카드이며 이 사건 호스트의 추가 증거가 아닙니다. 수집기의 고정 12항목에도 포함되지 않습니다.

### 모범 답안의 형태

“09:03 KST에 analyst의 sudo id 메시지와 동일 시각의 auid1000/euid0 Audit 이벤트가 있다. 높은 권한의 id 실행과 부합한다. 업무 승인과 당시 정책 전체는 제공되지 않았다. 정책·변경 요청·세션 연결 자료를 추가 확인하고, 업무 범위를 넘는 위임이 확인되면 최소 권한으로 조정한다.”

이 문장은 관찰·의미·누락·다음 조치를 담지만 침해 성공을 꾸며내지 않습니다. 또한 09:05 collector 서비스가 앞선 로그인에 의해 시작되었다는 인과관계는 자료만으로 확정할 수 없으므로 별도 가설로 남깁니다.

### 역할별 채점

기존 100점 루브릭의 교차 분석·반례 25점을 다음과 같이 적용합니다: Red Team 목적·전제 8점, Blue Team 아티팩트·수집 조건 8점, 정상 반례·한계·탐지 평가 9점. 새로 25점을 더하는 것이 아닙니다. bash 테스트 통과만으로 이 점수를 자동 부여하지 않습니다.

**완료 기준:** 세 카드 모두 여덟 연결 단계를 작성하고, GTFOBins 미등재를 안전 판정으로 쓰지 않으며, 미수집과 사건 부재를 구분합니다. 상대 역할의 결론 한 개를 근거로 수정한 기록을 제출합니다. 명령 실행 성공이나 그럴듯한 공격 시나리오보다 검증 가능한 설명을 평가합니다.


## 역할별 분석 기록

같은 실행 결과로 아래 항목을 작성하고 상대 관점에서 검토합니다. 자동 테스트는 계산과 원본 보존만 확인하며 이 서술 과제는 강사 또는 동료 검토 대상입니다.

| 항목 | 학생 작성 |
|---|---|
| Red Team 목적·필요 조건 | 관찰에서 도출한 질문과 전제 |
| 실제 관찰 | 파일·행·이벤트 ID와 출력 |
| Artifact·로깅 전제 | 확보한 자료와 필요한 기록 기능 |
| Blue Team 조사 | 정상 반례·추가 근거·수집 한계 |
| 탐지·완화 | 필요한 필드·오탐 사례·확인된 원인에 맞는 조치 |


## Checks

각 STEP의 test는 고정 자료의 계산 결과를 검사합니다. 아래는 원본 내용 보존을 확인합니다. 실행 성공과 침해 판정은 다릅니다. 어떤 결과가 사실이고 어떤 결론이 가설인지 교안 질문에 답합니다.


In [ ]:
after = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
assert before == after
print('원본 내용 보존: PASS')
print('분석 결과 파일 수:', sum(p.is_file() for p in output.rglob('*')))


## Next Steps

교안의 완료 기준에 따라 근거·정상 행위 가능성·누락·추가 확인을 제출합니다. 결과는 검토용 임시 폴더에 남습니다. 재실행은 Setup부터 새 폴더에서 시작하며 실제 증거를 공개 저장소에 올리지 않습니다.
